**Weather ingestion: control-table-driven design**

Rather than looping over a hardcoded list of cities and calling the API directly, this notebook:
1. Builds a control table (`control_weather_ingestion`) with one row per (city, year-month) chunk that needs loading
2. Runs an ingestion loop that reads only `pending`/`failed` rows from the control table, attempts each API call with retry/backoff, and updates that row's status immediately after each attempt

This makes reruns naturally idempotent — already-`success` rows are skipped automatically — and gives an auditable record of what loaded, when, and why anything failed, rather than relying on ad-hoc file-existence checks.

**Step 1: Identify target cities**

Using order volume from `silver_orders` joined to `silver_customers`, matched to lat/long centroids from `silver_geolocation`, to pick the top 15 cities that actually matter to the business — not all ~19,000 zip codes.

In [11]:
from pyspark.sql.functions import col, desc

df_customers_silver = spark.table("silver_customers")
df_orders_silver = spark.table("silver_orders")
df_geo_silver = spark.table("silver_geolocation")

top_cities = (
    df_orders_silver
    .join(df_customers_silver, "customer_id")
    .groupBy("customer_city", "customer_zip_code_prefix")
    .count()
    .orderBy(desc("count"))
    .limit(15)
)

top_cities_with_coords = (
    top_cities.join(
        df_geo_silver,
        top_cities.customer_zip_code_prefix == df_geo_silver.geolocation_zip_code_prefix,
        "left"
    )
    .select(
        col("customer_city").alias("city"),
        col("customer_zip_code_prefix").alias("zip_code_prefix"),
        col("avg_lat").alias("latitude"),
        col("avg_lng").alias("longitude")
    )
)

top_cities_with_coords.show(15, truncate=False)

StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 13, Finished, Available, Finished, False)

+--------------+---------------+-------------------+-------------------+
|city          |zip_code_prefix|latitude           |longitude          |
+--------------+---------------+-------------------+-------------------+
|rio de janeiro|22790          |-23.012986504226507|-43.46551954103294 |
|niteroi       |24220          |-22.90319701213708 |-43.1069203735102  |
|niteroi       |24220          |-22.903564689560653|-43.1077143031609  |
|rio de janeiro|22793          |-23.00061899109575 |-43.40471828456726 |
|niteroi       |24230          |-22.906462322547657|-43.106261077106026|
|niteroi       |24230          |-22.87481774137317 |-43.083328410369965|
|rio de janeiro|22775          |-22.969148201152674|-43.383065237968154|
|vila velha    |29101          |-20.345030195796227|-40.287903741996516|
|jundiai       |13212          |-23.176665062400335|-46.97401954835945 |
|jundiai       |13212          |-23.1747078166345  |-46.975020780226394|
|ipatinga      |35162          |-19.46878672545066 

**Step 2: Build the control table**

One row per (city, year-month) combination across the Olist order date range (Sep 2016 – Oct 2018, ~26 months). Cross-joining 15 cities × 26 months = ~390 chunks, each tracked individually.

In [19]:
from pyspark.sql.functions import lit, explode, sequence, to_date, date_format
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

# Generate year-month list: 2016-09 through 2018-10
months_df_raw = spark.sql("""
    SELECT explode(sequence(to_date('2016-09-01'), to_date('2018-10-01'), interval 1 month)) as month_date
""")

months_df = months_df_raw.select(date_format(col("month_date"), "yyyy-MM").alias("year_month"))

months_df.show(30, truncate=False)


cities_pd = top_cities_with_coords.toPandas()
months_pd = months_df.toPandas()

# Cross join in pandas (small: 15 x 26 = 390 rows, fine for driver-side processing)
import itertools
control_rows = []
for _, city_row in cities_pd.iterrows():
    for _, month_row in months_pd.iterrows():
        control_rows.append({
            "city": city_row["city"],
            "zip_code_prefix": str(city_row["zip_code_prefix"]),
            "latitude": float(city_row["latitude"]) if city_row["latitude"] is not None else None,
            "longitude": float(city_row["longitude"]) if city_row["longitude"] is not None else None,
            "year_month": month_row["year_month"],
            "status": "pending",
            "attempt_count": 0,
            "last_attempted_at": None,
            "rows_ingested": None,
            "error_message": None,
            "bronze_file_path": None
        })

control_schema = StructType([
    StructField("city", StringType(), True),
    StructField("zip_code_prefix", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("year_month", StringType(), True),
    StructField("status", StringType(), True),
    StructField("attempt_count", IntegerType(), True),
    StructField("last_attempted_at", TimestampType(), True),
    StructField("rows_ingested", IntegerType(), True),
    StructField("error_message", StringType(), True),
    StructField("bronze_file_path", StringType(), True),
])

df_control = spark.createDataFrame(control_rows, schema=control_schema)

print("Control table rows to create:", df_control.count())
print("Cities with missing coordinates:", df_control.filter(col("latitude").isNull()).select("city").distinct().count())

df_control.write.format("delta").mode("overwrite").saveAsTable("control_weather_ingestion")
print("control_weather_ingestion table created.")

StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 21, Finished, Available, Finished, False)

+----------+
|year_month|
+----------+
|2016-09   |
|2016-10   |
|2016-11   |
|2016-12   |
|2017-01   |
|2017-02   |
|2017-03   |
|2017-04   |
|2017-05   |
|2017-06   |
|2017-07   |
|2017-08   |
|2017-09   |
|2017-10   |
|2017-11   |
|2017-12   |
|2018-01   |
|2018-02   |
|2018-03   |
|2018-04   |
|2018-05   |
|2018-06   |
|2018-07   |
|2018-08   |
|2018-09   |
|2018-10   |
+----------+

Control table rows to create: 390
Cities with missing coordinates: 0
control_weather_ingestion table created.


In [13]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

geo_dedup_window = Window.partitionBy("geolocation_zip_code_prefix").orderBy("geolocation_city")
df_geo_canonical = (
    df_geo_silver
    .withColumn("rn", row_number().over(geo_dedup_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Geolocation rows before dedup:", df_geo_silver.count())
print("Geolocation rows after canonical dedup:", df_geo_canonical.count())

StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 15, Finished, Available, Finished, False)

Geolocation rows before dedup: 27912
Geolocation rows after canonical dedup: 19015


In [18]:
df_geo_silver.filter(col("geolocation_zip_code_prefix").isin("24220", "13212", "38400")).show(truncate=False)


StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 20, Finished, Available, Finished, False)

+---------------------------+----------------+-----------------+-------------------+-------------------+
|geolocation_zip_code_prefix|geolocation_city|geolocation_state|avg_lat            |avg_lng            |
+---------------------------+----------------+-----------------+-------------------+-------------------+
|13212                      |jundiaí         |SP               |-23.1747078166345  |-46.975020780226394|
|13212                      |jundiai         |SP               |-23.176665062400335|-46.97401954835945 |
|24220                      |niterói         |RJ               |-22.903564689560653|-43.1077143031609  |
|24220                      |niteroi         |RJ               |-22.90319701213708 |-43.1069203735102  |
|38400                      |uberlândia      |MG               |-18.91293835578346 |-48.27679777028317 |
|38400                      |uberlandia      |MG               |-18.913314060095804|-48.27851471068446 |
+---------------------------+----------------+---------

In [15]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

geo_dedup_window = Window.partitionBy("geolocation_zip_code_prefix").orderBy("geolocation_city")
df_geo_canonical = (
    df_geo_silver
    .withColumn("rn", row_number().over(geo_dedup_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

print("Geolocation rows before dedup:", df_geo_silver.count())
print("Geolocation rows after canonical dedup:", df_geo_canonical.count())

StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 17, Finished, Available, Finished, False)

Geolocation rows before dedup: 27912
Geolocation rows after canonical dedup: 19015


In [16]:
top_cities_with_coords = (
    top_cities.join(
        df_geo_canonical,
        top_cities.customer_zip_code_prefix == df_geo_canonical.geolocation_zip_code_prefix,
        "left"
    )
    .select(
        col("customer_city").alias("city"),
        col("customer_zip_code_prefix").alias("zip_code_prefix"),
        col("avg_lat").alias("latitude"),
        col("avg_lng").alias("longitude")
    )
)

top_cities_with_coords.show(15, truncate=False)

StatementMeta(, f0c331da-285b-4bd8-87b4-432d0f0c4773, 18, Finished, Available, Finished, False)

+--------------+---------------+-------------------+-------------------+
|city          |zip_code_prefix|latitude           |longitude          |
+--------------+---------------+-------------------+-------------------+
|rio de janeiro|22790          |-23.012986504226507|-43.46551954103294 |
|niteroi       |24220          |-22.90319701213708 |-43.1069203735102  |
|rio de janeiro|22793          |-23.00061899109575 |-43.40471828456726 |
|niteroi       |24230          |-22.87481774137317 |-43.083328410369965|
|rio de janeiro|22775          |-22.969148201152674|-43.383065237968154|
|vila velha    |29101          |-20.345030195796227|-40.287903741996516|
|jundiai       |13212          |-23.176665062400335|-46.97401954835945 |
|ipatinga      |35162          |-19.46878672545066 |-42.563894781305024|
|rio de janeiro|22631          |-23.00365666838402 |-43.34075885321382 |
|uberlandia    |38400          |-18.913314060095804|-48.27851471068446 |
|divinopolis   |35500          |-20.14061727686089 